In [1]:
import re, os
import sys
sys.path.insert(0, '.')

from openai_class import OpenAICompatibelClient
from prompt_engineering import AGENT_SYSTEM_PROMPT
from tools import available_tools

# 配置llm客户端
API_KEY = "sk-63bdc2c640c04adfb2822464b36cef43"
BASE_URL = "https://api.deepseek.com/v1"
MODEL_ID = "deepseek-chat"
TAVILY_API_KEY = "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"
os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

llm = OpenAICompatibelClient(model = MODEL_ID,api_key = API_KEY,base_url = BASE_URL)
user_prompt = "你好，请帮我查询一下今天成都的天气，然后根据天气推荐一个合适的旅游景点。"
prompt_history = ["用户请求：%s" % user_prompt]
print("用户输入：%s\n"%user_prompt + "="*40)

i = 0
print(f"--- 循环 {i+1} ---\n")
    
# 3.1. 构建Prompt
full_prompt = "\n".join(prompt_history)

# 3.2. 调用LLM进行思考
llm_output = llm.generate(full_prompt, system_prompt=AGENT_SYSTEM_PROMPT)
# 模型可能会输出多余的Thought-Action，需要截断
match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', llm_output, re.DOTALL)
if match:
    truncated = match.group(1).strip()
    if truncated != llm_output.strip():
        llm_output = truncated
        print("已截断多余的 Thought-Action 对")
print(f"模型输出:\n{llm_output}\n")
prompt_history.append(llm_output)

# 3.3. 解析并执行行动
action_match = re.search(r"Action：(.*)", llm_output, re.DOTALL)
if not action_match:
    observation = "错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ... Action: ...' 的格式。"
    observation_str = f"Observation: {observation}"
    print(f"{observation_str}\n" + "="*40)
    prompt_history.append(observation_str)
    #continue
action_str = action_match.group(1).strip()

if action_str.startswith("Finish"):
    final_answer = re.match(r"Finish\[(.*)\]", action_str).group(1)
    print(f"任务完成，最终答案: {final_answer}")
    #break

tool_name = re.search(r"(\w+)\(", action_str).group(1)
args_str = re.search(r"\((.*)\)", action_str).group(1)
kwargs = dict(re.findall(r'(\w+)="([^"]*)"', args_str))

if tool_name in available_tools:
    observation = available_tools[tool_name](**kwargs)
else:
    observation = f"错误:未定义的工具 '{tool_name}'"

# 3.4. 记录观察结果
observation_str = f"Observation: {observation}"
print(f"{observation_str}\n" + "="*40)
prompt_history.append(observation_str)

用户输入：你好，请帮我查询一下今天成都的天气，然后根据天气推荐一个合适的旅游景点。
--- 循环 1 ---

正在调用LLM
调用成功
模型输出:
Thought：用户需要查询成都的天气，并根据天气推荐旅游景点。我需要先使用get_weather工具获取成都的天气信息。
Action：get_weather(city="成都")

Observation: 成都当前天气:Light rain shower，气温13摄氏度


In [2]:
llm_output

'Thought：用户需要查询成都的天气，并根据天气推荐旅游景点。我需要先使用get_weather工具获取成都的天气信息。\nAction：get_weather(city="成都")'

NameError: name 'get_attraction' is not defined

In [4]:
import requests,os,re
from tavily import TavilyClient

def get_weather(city:str) -> str:

    #api端点，请求json格式的数据
    url = f"https://wttr.in/{city}?format=j1"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        current_condition = data['current_condition'][0]
        weather_desc = current_condition['weatherDesc'][0]['value']
        temp_c = current_condition['temp_C']

        return f"{city}当前天气:{weather_desc}，气温{temp_c}摄氏度"
    
    except requests.exceptions.RequestException as e:
        # 处理网络错误
        return f"错误:查询天气时遇到网络问题 - {e}"
    except (KeyError, IndexError) as e:
        # 处理数据解析错误
        return f"错误:解析天气数据失败，可能是城市名称无效 - {e}"


def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"
    
    # 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=api_key)

    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"

    try:
        # 4. 调用API，include_answer=True会返回一个综合性的回答
        response = tavily.search(query=query, search_depth="basic", include_answer=True)

        # 5. Tavily返回的结果已经非常干净，可以直接使用
        # response['answer'] 是一个基于所有搜索结果的总结性回答
        if response.get("answer"):
            return response["answer"]
        
        # 如果没有综合性回答，则格式化原始结果
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")
        
        if not formatted_results:
             return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"



In [5]:
get_attraction("成都","Light rain shower，气温13摄氏度")

"In light rain and 13°C weather, Chengdu's Panda Base is ideal for viewing pandas. It's clean and quiet, perfect for a pleasant visit."